# Clickbait Spoiling - Deep Dual-Transformer Pipeline

This notebook orchestrates the end-to-end training and evaluation pipeline for the clickbait spoiling task.
The architecture leverages a RoBERTa-based Classifier for Task 1 (Spoiler Type Classification) and a
Fine-tuned Extractive Question-Answering Transformer for Task 2 (Spoiler Text Generation).


In [ ]:
import os
import sys


def setup_kaggle_environment():
    """Clones or updates the project code repository and installs required production packages."""
    working_dir = "/kaggle/working"
    repo_name = "clickbait-spoiling-IR-final-project"
    repo_dir = os.path.join(working_dir, repo_name)
    repo_url = "https://github.com/aryonmt/clickbait-spoiling-IR-final-project"

    # Synchronize codebase from remote repository
    if not os.path.exists(repo_dir):
        print(f"[INFO] Cloning code repository from {repo_url}...")
        os.chdir(working_dir)
        os.system(f"git clone {repo_url}")
    else:
        print("[INFO] Repository already exists. Pulling latest updates...")
        os.chdir(repo_dir)
        os.system("git pull origin main")

    # Navigate to the repository root directory
    os.chdir(repo_dir)
    sys.path.append(repo_dir)

    # Ingest specialized deep learning and text processing dependencies
    print("[INFO] Installing external environment dependencies...")
    os.system(
        "pip install -q transformers[torch] datasets accelerate evaluate scikit-learn rouge-score nltk"
    )
    print("[SUCCESS] Environment initialization completed.")


setup_kaggle_environment()

## Phase 1: Task 1 Model Training (Spoiler Classification)

Fine-tunes a `roberta-base` architecture utilizing dynamic class weighting to mitigate severe distribution imbalances.


In [ ]:
!python main_transformer_task1.py \

    --train_path "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl" \

    --val_path "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl" \

    --model_name "roberta-base" \

    --lr 1e-5 \

    --output_dir "/kaggle/working/results_roberta"

## Phase 2: Task 2 Model Training (Extractive QA)

Fine-tunes a pre-trained QA checkpoint (`deepset/roberta-base-squad2`) to predict precise answer token index spans.


In [ ]:
!python main_transformer_task2_qa.py \

    --train_path "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl" \

    --val_path "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl" \

    --model_name "deepset/roberta-base-squad2" \

    --output_dir "/kaggle/working/results_qa" \

    --lr 2e-5

## Phase 3: End-to-End Hybrid Pipeline Inference

Combines the discrete outputs of both transformers to generate the final formatted submission file and prints full validation metrics.


In [ ]:
!python main_hybrid_pipeline.py \

    --val_path "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl" \

    --t1_pred_path "/kaggle/input/datasets/amirrezanemati05/task1-predictions/run_transformer_baseline (1).jsonl" \

    --t2_checkpoint "/kaggle/working/results_qa/checkpoint-1218" \

    --output_path "/kaggle/working/final_complete_submission.jsonl"